# SoftCAM-Transformer v5 — Run D1 (comparaison MLP vs dot_product)

**Architecture v5 (commune aux deux variantes) :**
```
M           = softmax(score(dec, enc))                       ← carte d'évidence [B, P, C]
h_evidence  = LayerNorm(bmm(M, enc_hidden))                  ← "sortie de la carte" [B, P, D]
h_ev        = dec_output * (1 + alpha * tanh(h_evidence))    ← PRODUIT borné
```

**Deux modes de calcul de M (le seul changement entre les deux runs) :**

| Mode | Formule du score | Paramètres |
|--|--|--|
| `mlp` | `score[t,c] = MLP(dec_output[t])[c]` | 7920 |
| `dot_product` | `score[t,c] = (dec_output[t] @ W_q) · (enc_hidden[c] @ W_k) / √D` | 2048 |

**Pourquoi cette comparaison :**

Les expériences C1/C2/C3 ont échoué à cause de la multiplicativité non bornée (gate ∈ (0, 2)).
On garde la multiplicativité (demandée par les encadreurs) mais **bornée** par alpha petit (~0.1)
→ gate ∈ (0.9, 1.1) → stable en autorégression.

En faisant tourner les deux modes avec **le même seed et les mêmes hyperparams**, on isole
l'effet du calcul de M : si dot_product > mlp → les encadreurs avaient raison.

**Schedule** (analogue B5) :
- α : 0 (epochs 0-14, warm-up Transformer seul) → ramp 0→0.1 (15-24) → plateau 0.1 (25+)
- γ entropie : 0 (epochs 0-24) → ramp 0→1e-3 (25-39) → plateau 1e-3 (40+)

**Coût** : ~2h Colab T4 (2 × 51 epochs).

**Cible** : R² ≥ 0.30 (GATE H1.C). Comparaison avec B5 (0.7563) en synthèse finale.

> Avant de lancer : Runtime → T4 GPU.

## 1 — Setup

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'Pas de GPU — vérifier le runtime.')

In [ ]:
%%capture
!pip install -q transformers datasets "gluonts[torch]" accelerate evaluate scipy scikit-learn tqdm openpyxl ujson

## 2 — Récupération du code

In [ ]:
import os, sys

REPO_URL = 'https://github.com/theblackmamba08/recherche-m2-xai-faas.git'
REPO_DIR = '/content/recherche-m2-xai-faas'

ipy = get_ipython()
if not os.path.isdir(REPO_DIR):
    ipy.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    ipy.system(f'git -C {REPO_DIR} pull')

if f'{REPO_DIR}/code' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/code')

print('Repo prêt.')

## 3 — Imports + hyperparams

In [ ]:
import random, json, gc, copy
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from functools import lru_cache, partial
from pathlib import Path

SEED = 998
def reset_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

reset_seed()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')

# ── Hyperparams (identiques B5) ───────────────────────────────────────────────
FREQ              = '1T'
PREDICTION_LENGTH = 120
CONTEXT_LENGTH    = 240
D_MODEL           = 32
N_HEADS           = 2
ENCODER_LAYERS    = 4
DECODER_LAYERS    = 4
EMBEDDING_DIM     = [2]
DROPOUT           = 0.1
BATCH_SIZE_TRAIN  = 256
BATCH_SIZE_TEST   = 64
NUM_BATCHES_EPOCH = 100
LR                = 6e-4
BETAS             = (0.9, 0.95)
WEIGHT_DECAY      = 1e-1
GRAD_CLIP_NORM    = 1.0
NUM_EPOCHS        = 51

USE_EVIDENCE_LAYER   = True
ALPHA_L1             = 0.0
BETA_L2              = 1e-3
GAMMA_ENTROPY_TARGET = 1e-3

# ── Schedule α (amplitude du produit borné) ──────────────────────────────────
ALPHA_TARGET     = 0.1     # plateau final (gate ∈ (1-0.1, 1+0.1))
ALPHA_WARMUP_END = 15      # epochs 0-14 : α=0 (Transformer seul, comme B5)
ALPHA_RAMP_END   = 25      # epochs 15-24 : ramp 0→0.1

def alpha_schedule(epoch: int) -> float:
    if epoch < ALPHA_WARMUP_END:
        return 0.0
    if epoch < ALPHA_RAMP_END:
        return ALPHA_TARGET * (epoch - ALPHA_WARMUP_END) / (ALPHA_RAMP_END - ALPHA_WARMUP_END)
    return ALPHA_TARGET

# ── Schedule γ entropie ───────────────────────────────────────────────────────
GAMMA_ANNEAL_START = 25
GAMMA_ANNEAL_END   = 40

def gamma_schedule(epoch: int) -> float:
    if epoch < GAMMA_ANNEAL_START:
        return 0.0
    if epoch < GAMMA_ANNEAL_END:
        return GAMMA_ENTROPY_TARGET * (epoch - GAMMA_ANNEAL_START) / (GAMMA_ANNEAL_END - GAMMA_ANNEAL_START)
    return GAMMA_ENTROPY_TARGET

# ── Références ────────────────────────────────────────────────────────────────
FAYAM_R2, FAYAM_SPEAR = 0.3701, 0.9201
B5_R2,    B5_SPEAR    = 0.7563, 0.9870
C1_R2,    C2_R2       = -5.2913, -5.3643
GATE_R2,  GATE_SPEAR  = 0.30, 0.85

CLUSTER_ID = 4
M_MODES = ['mlp', 'dot_product']

print(f'Schedule α : 0..{ALPHA_WARMUP_END-1}→0  {ALPHA_WARMUP_END}..{ALPHA_RAMP_END-1}→ramp  {ALPHA_RAMP_END}+→{ALPHA_TARGET}')
print(f'Schedule γ : 0..{GAMMA_ANNEAL_START-1}→0  {GAMMA_ANNEAL_START}..{GAMMA_ANNEAL_END-1}→ramp  {GAMMA_ANNEAL_END}+→{GAMMA_ENTROPY_TARGET:.0e}')
print(f'Modes      : {M_MODES}')

## 4 — Visualisation schedules

In [ ]:
ep_axis     = np.arange(NUM_EPOCHS)
alpha_vals  = np.array([alpha_schedule(e) for e in ep_axis])
gamma_vals  = np.array([gamma_schedule(e) for e in ep_axis])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(ep_axis, alpha_vals, color='#1f77b4', lw=2.5, label='alpha (amplitude produit)')
ax1.axvline(ALPHA_WARMUP_END, color='gray', ls='--', alpha=0.5, label=f'warm-up end (e={ALPHA_WARMUP_END})')
ax1.axvline(ALPHA_RAMP_END,   color='gray', ls=':',  alpha=0.5, label=f'plateau (e={ALPHA_RAMP_END})')
ax1.axhline(ALPHA_TARGET, color='gray', ls=':', alpha=0.3)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('alpha')
ax1.set_title('Schedule alpha (warm-up + plateau bas)')
ax1.legend(loc='lower right'); ax1.grid(alpha=0.3)

ax2.plot(ep_axis, gamma_vals, color='firebrick', lw=2.5, label='gamma (entropie M)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('gamma')
ax2.set_title('Schedule gamma')
ax2.legend(loc='lower right'); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5 — Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/m2-xai-faas/experiments/softcam-cluster4-v5-runD1-comparison'
for mode in M_MODES:
    for subdir in ['checkpoints', 'results', 'figures']:
        os.makedirs(f'{DRIVE_BASE}/{mode}/{subdir}', exist_ok=True)
print(f'Dossier comparaison : {DRIVE_BASE}')

## 6 — Chargement Cluster 4

In [ ]:
from datasets import Dataset

DATA_DIR   = Path('/content/drive/MyDrive/Recherche/Datasets')
START_DATE = '2021-01-01 00:00:00'

df = pd.read_csv(DATA_DIR / f'cluster_{CLUSTER_ID}.csv', index_col='Function')
all_series = []
for func_id, row in df.iterrows():
    all_series.append({
        'function_id': int(func_id),
        'cluster':     CLUSTER_ID,
        'target_full': row.values.astype(np.float32),
    })

train_rows, test_rows = [], []
for ts_idx, s in enumerate(all_series):
    base = {'start': START_DATE, 'feat_static_cat': [ts_idx],
            'cluster': s['cluster'], 'function_id': s['function_id']}
    train_rows.append({**base, 'target': s['target_full'][:-PREDICTION_LENGTH].tolist()})
    test_rows.append( {**base, 'target': s['target_full'].tolist()})

train_dataset = Dataset.from_list(train_rows)
test_dataset  = Dataset.from_list(test_rows)
print(f'Cluster {CLUSTER_ID} — train={len(train_dataset)}  test={len(test_dataset)}')

## 7 — Pipeline GluonTS

In [ ]:
from gluonts.time_feature import get_lags_for_frequency, time_features_from_frequency_str
from gluonts.dataset.field_names import FieldName
from gluonts.transform import (
    AddAgeFeature, AddObservedValuesIndicator, AddTimeFeatures,
    AsNumpyArray, Chain, ExpectedNumInstanceSampler, InstanceSplitter,
    RemoveFields, ValidationSplitSampler,
    VstackFeatures, RenameFields,
)
from gluonts.itertools import Cyclic, Cached
from gluonts.dataset.loader import as_stacked_batches

lags_sequence = get_lags_for_frequency(FREQ)
time_features = time_features_from_frequency_str(FREQ)

@lru_cache(10_000)
def convert_to_pandas_period(date, freq):
    return pd.Period(date, freq)

def transform_start_field(batch, freq):
    batch['start'] = [convert_to_pandas_period(d, freq) for d in batch['start']]
    return batch

for ds in (train_dataset, test_dataset):
    ds.set_transform(partial(transform_start_field, freq=FREQ))

def create_transformation(freq, config):
    remove_field_names = []
    if config.num_static_real_features == 0:
        remove_field_names.append(FieldName.FEAT_STATIC_REAL)
    if config.num_dynamic_real_features == 0:
        remove_field_names.append(FieldName.FEAT_DYNAMIC_REAL)
    if config.num_static_categorical_features == 0:
        remove_field_names.append(FieldName.FEAT_STATIC_CAT)
    return Chain(
        [RemoveFields(field_names=remove_field_names)]
        + ([AsNumpyArray(field=FieldName.FEAT_STATIC_CAT, expected_ndim=1, dtype=int)]
           if config.num_static_categorical_features > 0 else [])
        + [AsNumpyArray(field=FieldName.TARGET, expected_ndim=1),
           AddObservedValuesIndicator(target_field=FieldName.TARGET,
                                      output_field=FieldName.OBSERVED_VALUES),
           AddTimeFeatures(start_field=FieldName.START, target_field=FieldName.TARGET,
                           output_field=FieldName.FEAT_TIME,
                           time_features=time_features_from_frequency_str(freq),
                           pred_length=config.prediction_length),
           AddAgeFeature(target_field=FieldName.TARGET, output_field=FieldName.FEAT_AGE,
                         pred_length=config.prediction_length, log_scale=True),
           VstackFeatures(output_field=FieldName.FEAT_TIME,
                          input_fields=[FieldName.FEAT_TIME, FieldName.FEAT_AGE]),
           RenameFields(mapping={
               FieldName.FEAT_STATIC_CAT: 'static_categorical_features',
               FieldName.FEAT_TIME:       'time_features',
               FieldName.TARGET:          'values',
               FieldName.OBSERVED_VALUES: 'observed_mask',
           })]
    )

def create_instance_splitter(config, mode):
    sampler = {
        'train':      ExpectedNumInstanceSampler(num_instances=1.0, min_future=config.prediction_length),
        'validation': ValidationSplitSampler(min_future=config.prediction_length),
    }[mode]
    return InstanceSplitter(
        target_field='values', is_pad_field=FieldName.IS_PAD,
        start_field=FieldName.START, forecast_start_field=FieldName.FORECAST_START,
        instance_sampler=sampler,
        past_length=config.context_length + max(config.lags_sequence),
        future_length=config.prediction_length,
        time_series_fields=['time_features', 'observed_mask'],
    )

def _pred_fields(config):
    f = ['past_time_features', 'past_values', 'past_observed_mask', 'future_time_features']
    if config.num_static_categorical_features > 0:
        f.append('static_categorical_features')
    return f

def create_train_dataloader(config, freq, data, batch_size, num_batches_per_epoch):
    tr = create_transformation(freq, config)
    sp = create_instance_splitter(config, 'train')
    return as_stacked_batches(
        Cyclic(Cached(sp.apply(tr.apply(data, is_train=True), is_train=True))),
        batch_size=batch_size,
        num_batches_per_epoch=num_batches_per_epoch,
        output_type=torch.tensor,
        field_names=['past_time_features', 'past_values', 'past_observed_mask',
                     'future_time_features', 'future_values', 'future_observed_mask']
        + (['static_categorical_features'] if config.num_static_categorical_features > 0 else []),
    )

def create_backtest_dataloader(config, freq, data, batch_size):
    tr = create_transformation(freq, config)
    sp = create_instance_splitter(config, 'validation')
    return as_stacked_batches(sp.apply(tr.apply(data), is_train=True),
                              batch_size=batch_size, output_type=torch.tensor,
                              field_names=_pred_fields(config))

print('Pipeline GluonTS prête.')

## 8 — Fonction d'entraînement + évaluation v5

In [ ]:
from src.models.softcam_transformer_v5 import (
    SoftCAMTransformerV5Config,
    SoftCAMTransformerV5ForPrediction,
)
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

def build_model_v5(m_mode):
    cfg = SoftCAMTransformerV5Config(
        m_mode=m_mode,
        prediction_length=PREDICTION_LENGTH,
        context_length=CONTEXT_LENGTH,
        lags_sequence=lags_sequence,
        num_time_features=len(time_features) + 1,
        num_static_categorical_features=1,
        cardinality=[len(train_dataset)],
        embedding_dimension=EMBEDDING_DIM,
        encoder_layers=ENCODER_LAYERS,
        decoder_layers=DECODER_LAYERS,
        d_model=D_MODEL,
        encoder_attention_heads=N_HEADS,
        decoder_attention_heads=N_HEADS,
        encoder_ffn_dim=max(D_MODEL, 32),
        decoder_ffn_dim=max(D_MODEL, 32),
        dropout=DROPOUT,
        use_evidence_layer=USE_EVIDENCE_LAYER,
        evidence_mix=0.05,
        alpha_l1=ALPHA_L1,
        beta_l2=BETA_L2,
        gamma_entropy=GAMMA_ENTROPY_TARGET,
    )
    model = SoftCAMTransformerV5ForPrediction(cfg).to(device)
    model.ev_layer_v5.alpha = 0.0  # init warm-up
    return cfg, model

def train_v5(m_mode):
    print(f'\n{"="*70}\nEntraînement v5 mode={m_mode!r}\n{"="*70}')
    reset_seed(SEED)
    cfg, model = build_model_v5(m_mode)

    n_params = sum(p.numel() for p in model.parameters())
    n_ev_params = sum(p.numel() for p in model.ev_layer_v5.parameters())
    print(f'  Paramètres totaux       : {n_params:,}')
    print(f'  Paramètres ev_layer_v5  : {n_ev_params:,}')

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=BETAS, weight_decay=WEIGHT_DECAY)
    train_loader = create_train_dataloader(cfg, FREQ, train_dataset, BATCH_SIZE_TRAIN, NUM_BATCHES_EPOCH)

    history = []
    for epoch in range(NUM_EPOCHS):
        alpha = alpha_schedule(epoch)
        gamma = gamma_schedule(epoch)
        model.ev_layer_v5.alpha = alpha
        model.gamma_entropy = gamma

        model.train()
        ep = {'total': [], 'forecast': [], 'entropy': []}
        for batch in train_loader:
            optimizer.zero_grad()
            out = model(
                static_categorical_features=batch['static_categorical_features'].to(device)
                    if cfg.num_static_categorical_features > 0 else None,
                past_time_features=batch['past_time_features'].to(device),
                past_values=batch['past_values'].to(device),
                past_observed_mask=batch['past_observed_mask'].to(device),
                future_time_features=batch['future_time_features'].to(device),
                future_values=batch['future_values'].to(device),
                future_observed_mask=batch['future_observed_mask'].to(device),
            )
            if out.loss is None:
                continue
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
            ep['total'].append(out.loss.item())
            if out.forecast_loss is not None: ep['forecast'].append(out.forecast_loss.item())
            if out.entropy_loss is not None:  ep['entropy'].append(out.entropy_loss.item())

        mloss = np.mean(ep['total'])
        mfc   = np.mean(ep['forecast']) if ep['forecast'] else float('nan')
        ment  = np.mean(ep['entropy'])  if ep['entropy']  else float('nan')
        history.append({'epoch': epoch, 'loss': mloss, 'forecast': mfc, 'entropy': ment,
                        'alpha': alpha, 'gamma': gamma})

        marker = ''
        if epoch == ALPHA_WARMUP_END:   marker = '  ← α ramp start'
        elif epoch == ALPHA_RAMP_END:   marker = '  ← α plein'
        elif epoch == GAMMA_ANNEAL_END: marker = '  ← γ plein'

        print(f'  ep {epoch:3d}  loss={mloss:.4f}  fc={mfc:.4f}  H={ment:.4f}  α={alpha:.3f}  γ={gamma:.1e}{marker}')

    # Forcer α=ALPHA_TARGET pour l'inférence
    model.ev_layer_v5.alpha = ALPHA_TARGET

    # Évaluation
    test_loader = create_backtest_dataloader(cfg, FREQ, test_dataset, BATCH_SIZE_TEST)
    model.eval()
    with torch.no_grad():
        forecasts = []
        for b in test_loader:
            out = model.generate(
                static_categorical_features=b['static_categorical_features'].to(device)
                    if cfg.num_static_categorical_features > 0 else None,
                past_time_features=b['past_time_features'].to(device),
                past_values=b['past_values'].to(device),
                future_time_features=b['future_time_features'].to(device),
                past_observed_mask=b['past_observed_mask'].to(device),
            )
            forecasts.append(out.sequences.cpu().numpy())
        forecasts = np.vstack(forecasts)
        forecast_median = np.median(forecasts, axis=1)

    r2s, spears = [], []
    for ts_idx in range(len(test_dataset)):
        target = np.array(test_dataset[ts_idx]['target'])
        actual = target[-PREDICTION_LENGTH:]
        pred   = forecast_median[ts_idx]
        r2s.append(r2_score(actual, pred))
        rho, _ = spearmanr(actual, pred)
        spears.append(rho)
    r2_mean    = float(np.mean(r2s))
    spear_mean = float(np.mean(spears))

    # Sauvegarde checkpoint
    ckpt_path = f'{DRIVE_BASE}/{m_mode}/checkpoints/v5_runD1_{m_mode}_final.pt'
    torch.save({
        'model': model.state_dict(),
        'config': cfg.to_dict(),
        'history': history,
        'm_mode': m_mode,
        'r2_mean': r2_mean,
        'spearman_mean': spear_mean,
        'r2_per_series': r2s,
        'spearman_per_series': spears,
    }, ckpt_path)

    print(f'\n  RÉSULTATS v5 {m_mode}:')
    print(f'    R²       = {r2_mean:+.4f}  (B5={B5_R2:.4f}  FAYAM={FAYAM_R2:.4f})')
    print(f'    Spearman = {spear_mean:+.4f}')
    print(f'    Per-series R² : {[f"{x:+.3f}" for x in r2s]}')
    print(f'    Checkpoint : {ckpt_path}')

    return {
        'm_mode': m_mode,
        'cfg': cfg,
        'history': history,
        'r2_mean': r2_mean,
        'spearman_mean': spear_mean,
        'r2_per_series': r2s,
        'spearman_per_series': spears,
        'n_params': n_params,
        'n_ev_params': n_ev_params,
    }

print('Fonction train_v5 prête.')

## 9 — Entraînement variante A : MLP

In [ ]:
result_mlp = train_v5('mlp')

## 10 — Entraînement variante B : dot_product

In [ ]:
result_dot = train_v5('dot_product')

## 11 — Comparaison + courbes

In [ ]:
results = [result_mlp, result_dot]

# Tableau récap
df_recap = pd.DataFrame([
    {'mode': r['m_mode'],
     'n_ev_params': r['n_ev_params'],
     'R²': r['r2_mean'],
     'Spearman': r['spearman_mean'],
     'gate H1.C': '✅' if (r['r2_mean']>=GATE_R2 and r['spearman_mean']>=GATE_SPEAR) else '❌',
     'vs B5': r['r2_mean'] - B5_R2,
     'vs FAYAM': r['r2_mean'] - FAYAM_R2}
    for r in results
])
print(df_recap.to_string(index=False))

# Courbes de loss
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
colors = {'mlp': '#1f77b4', 'dot_product': '#d62728'}

for r in results:
    eps = [h['epoch'] for h in r['history']]
    axes[0, 0].plot(eps, [h['loss']     for h in r['history']], color=colors[r['m_mode']], lw=2, label=r['m_mode'])
    axes[0, 1].plot(eps, [h['forecast'] for h in r['history']], color=colors[r['m_mode']], lw=2, label=r['m_mode'])
    axes[1, 0].plot(eps, [h['alpha']    for h in r['history']], color=colors[r['m_mode']], lw=2, label=r['m_mode'])
    axes[1, 1].plot(eps, [h['entropy']  for h in r['history']], color=colors[r['m_mode']], lw=2, label=r['m_mode'])

for ax, title in zip(axes.flat, ['Loss totale', 'Loss forecast (NLL StudentT)', 'alpha schedule', 'gamma * H(M)']):
    ax.set_xlabel('Epoch'); ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
    ax.axvline(ALPHA_WARMUP_END, color='gray', ls='--', alpha=0.4)
    ax.axvline(ALPHA_RAMP_END,   color='gray', ls=':',  alpha=0.4)

plt.suptitle('v5 RunD1 — Comparaison MLP vs dot_product (même seed, même hyperparams)')
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/comparison_curves.png', dpi=150)
plt.show()

## 12 — Verdict + sauvegarde JSON

In [ ]:
best = max(results, key=lambda r: r['r2_mean'])

summary = {
    'run': 'softcam-cluster4-v5-runD1-comparison',
    'description': 'Compare v5 mode=mlp vs mode=dot_product avec produit borné identique.',
    'seed': SEED,
    'num_epochs': NUM_EPOCHS,
    'alpha_target': ALPHA_TARGET,
    'alpha_warmup_end': ALPHA_WARMUP_END,
    'alpha_ramp_end': ALPHA_RAMP_END,
    'results': [
        {k: v for k, v in r.items() if k != 'cfg' and k != 'history'}
        for r in results
    ],
    'best_mode': best['m_mode'],
    'best_r2':   best['r2_mean'],
    'references': {
        'fayam':   {'r2': FAYAM_R2, 'spearman': FAYAM_SPEAR},
        'b5_v3':   {'r2': B5_R2,    'spearman': B5_SPEAR},
        'c1_v4':   {'r2': C1_R2},
        'c2_v4':   {'r2': C2_R2},
    },
}

with open(f'{DRIVE_BASE}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print('='*72)
print(f'  VERDICT v5 RunD1 (seed={SEED})')
print('='*72)
print(f'  MLP         : R²={result_mlp["r2_mean"]:+.4f}   ρ={result_mlp["spearman_mean"]:+.4f}')
print(f'  Dot product : R²={result_dot["r2_mean"]:+.4f}   ρ={result_dot["spearman_mean"]:+.4f}')
print()
print(f'  Meilleur    : {best["m_mode"]}  R²={best["r2_mean"]:+.4f}')
print(f'  vs B5       : {best["r2_mean"] - B5_R2:+.4f}')
print(f'  vs FAYAM    : {best["r2_mean"] - FAYAM_R2:+.4f}')
print()

# Lecture
r2_mlp = result_mlp['r2_mean']
r2_dot = result_dot['r2_mean']

if r2_mlp >= GATE_R2 and r2_dot >= GATE_R2:
    print('  ✅ Les deux variantes passent → produit borné v5 viable.')
    if abs(r2_dot - r2_mlp) < 0.05:
        print('     Pas de différence notable MLP vs dot product → choisir dot product')
        print('     (encadreurs satisfaits, paramètres moindres).')
    elif r2_dot > r2_mlp:
        print(f'     Dot product gagne par {r2_dot - r2_mlp:+.4f} pp → confirmation Vaswani.')
    else:
        print(f'     MLP gagne par {r2_mlp - r2_dot:+.4f} pp → dot product est moins adapté.')
elif max(r2_mlp, r2_dot) >= GATE_R2:
    winner = 'dot_product' if r2_dot > r2_mlp else 'mlp'
    print(f'  🟡 Seul {winner} passe GATE H1.C → adopter ce mode comme architecture finale.')
elif max(r2_mlp, r2_dot) > 0:
    print('  🟡 R² positifs mais < GATE → produit borné meilleur que v4 mais insuffisant.')
elif max(r2_mlp, r2_dot) > -1:
    print('  ⚠️ R² ≈ 0 → architecture limite, à itérer (alpha plus grand ? plus d\'epochs ?)')
else:
    print('  ❌ Échec systémique → produit multiplicatif inviable même borné.')
print('='*72)